# Notebook 03 — Non-Linearity Capacity (NLC)

**QNLEP metric:** NLC measures Fourier reach as a **continuous** score derived from
the full `R²(k)` curve, not only the largest passed threshold. Each method is trained
on `sin(kx)` for increasing `k`, and the score integrates **positive-clipped** fit
quality over the normalized `log2(k)` axis.

```
NLC = area_under_curve( clip(R²(k), 0, 1)  vs  normalized_log2(k) )
```

For reviewer-facing diagnostics, `results/nlc_detail.json` still exports the old
threshold-style summary (`legacy_k_star`, `legacy_threshold_score`) together with the
full `r2_by_k` curve and `positive_r2_by_k` contributions.

**Negative R².** `sklearn.metrics.r2_score` can be **negative**: the model is worse than
predicting the constant mean of `y`. That is normal for hard `k` (fast `sin(kx)`)
or too few training steps — not a numerical bug.

**Methods evaluated (see `QNLEP_METHOD_ORDER` in the config cell):** data re-uploading, ancilla tracing, **ancilla + re-uploading** (combined), strongly entangling layers, QCNN, quantum reservoir + readout — all wired via `src/qnlep_registry.py`. Edit the list to ablate.  
**Task suite:** `f_k(x) = sin(kx)`, `k ∈ {1, 2, 4, 8, 16}`, `x ∈ [-π, π]`  
**Outputs:** R² heatmap (method × k), NLC score per method, incremental
`results/nlc_results_partial.json` after each (method, k) run (resume-safe). Changing
The partial file embeds a **fingerprint** (seed, steps, `N_STEPS_BY_METHOD`, `(n,L)`, `K_VALUES`, etc.). If any of those change, the next run **ignores** the old file automatically. For a manual wipe: set **`NLC_LOAD_PARTIAL_FROM_DISK = False`**, env **`QNLEP_NLC_IGNORE_PARTIAL=1`**, **`NLC_FORCE_FULL_RETRAIN = True`**, or delete **`nlc_results_partial.json`**. Check **`[NLC partial] ...`** for `force_full` / `load_disk` / `drop`.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import json, pathlib
import os

from src.qnlep_registry import QNLEP_BUILDERS, param_shape
from src.qnlep_run_paths import protocol_results_dir, protocol_n_qubits, protocol_n_layers
from src.metrics.protocol_scores import (
    continuous_frequency_score,
    k_star_at_threshold as legacy_k_star_from_curve,
    positive_r2_score,
)

RUN_SEED = int(os.environ.get("QNLEP_SEED", "42"))
np.random.seed(RUN_SEED)
print("QNLEP_SEED =", RUN_SEED, "(set env QNLEP_SEED before Jupyter to replicate)")


## 1. Configuration

In [ ]:
K_VALUES   = [1, 2, 4, 8, 16]
K_MAX      = 16
R2_THRESH  = 0.95

N_TRAIN    = 36   # compact Fourier grid; continuous NLC is less threshold-fragile than the legacy score
N_EVAL     = 120
N_STEPS    = 50
LR         = 0.1
N_QUBITS   = protocol_n_qubits(3)
N_LAYERS   = protocol_n_layers(3)

# Extra steps for methods that usually need more budget on the higher-frequency probes.
# Changing steps/seed/(n,L)/method list auto-invalidates the partial JSON via embedded fingerprint.
N_STEPS_BY_METHOD = {
    "ancilla_reuploading": 100,
    "quantum_reservoir": 120,
}

# Partial JSON skips training when (method, k) already has 'r2' — so more steps
# do NOT apply until you clear those rows. Either delete `nlc_results_partial.json`,
# or use the switches below (same idea as NSS_FORCE_CLEAN_RETRAIN in notebook 07).
NLC_FORCE_FULL_RETRAIN = False  # True: ignore entire partial file this run
# Use a tuple OR a comma-separated string (not a bare string name — that would iterate by character).
NLC_FORCE_RETRAIN_METHODS = ()  # e.g. ("ancilla_reuploading", "quantum_reservoir")
# or: "ancilla_reuploading, quantum_reservoir"
NLC_LOAD_PARTIAL_FROM_DISK = True  # False: never read partial JSON (full retrain; file still updated after each k)

RESULTS_DIR = protocol_results_dir()
FIGURES_DIR = pathlib.Path('../figures')
NLC_PARTIAL_JSON = RESULTS_DIR / 'nlc_results_partial.json'
for p in (RESULTS_DIR, FIGURES_DIR):
    p.mkdir(parents=True, exist_ok=True)

x_train = np.linspace(-np.pi, np.pi, N_TRAIN)
x_eval  = np.linspace(-np.pi, np.pi, N_EVAL)

# Full QNLEP method list (see src/qnlep_registry.py). Edit to ablate methods.
QNLEP_METHOD_ORDER = ['data_reuploading', 'ancilla_tracing', 'ancilla_reuploading', 'strongly_entangling', 'qcnn', 'quantum_reservoir']

METHODS = {}
for name in QNLEP_METHOD_ORDER:
    if name == "quantum_reservoir":
        # Frozen reservoir unitary depends on seed — tie to RUN_SEED for reproducible multi-seed studies.
        METHODS[name] = QNLEP_BUILDERS[name](N_QUBITS, N_LAYERS, reservoir_seed=RUN_SEED)
    else:
        METHODS[name] = QNLEP_BUILDERS[name](N_QUBITS, N_LAYERS)


## 2. Training helper

In [ ]:
def train_on_task(method_name, circuit, x_tr, y_tr, n_steps=N_STEPS, lr=LR, seed=None):
    """Train circuit on (x_tr, y_tr). Returns (weights, loss_history)."""
    shape = param_shape(method_name, N_QUBITS, N_LAYERS)
    rng = np.random.default_rng(RUN_SEED if seed is None else seed)
    weights = pnp.array(rng.normal(0, 0.1, shape), requires_grad=True)

    def cost(w):
        preds = pnp.array([circuit(w, float(x)) for x in x_tr])
        return pnp.mean((preds - y_tr) ** 2)

    opt = qml.AdamOptimizer(stepsize=lr)
    history = []
    for _ in range(n_steps):
        weights, loss = opt.step_and_cost(cost, weights)
        history.append(float(loss))
    return weights, history


def evaluate(circuit, weights, x_ev):
    """Return predictions on evaluation grid."""
    return np.array([float(circuit(weights, float(x))) for x in x_ev])


## 3. Sweep: train each method on each sin(kx)

In [ ]:
import os
import sys


def _nlc_config_get(name, default):
    """Notebook config lives in another cell; check this cell's globals() then Jupyter __main__."""
    g = globals()
    if name in g:
        return g[name]
    m = sys.modules.get("__main__")
    if m is not None and hasattr(m, name):
        return getattr(m, name)
    return default


def _partial_to_disk(obj, fingerprint):
    body = {m: {str(k): v for k, v in d.items()} for m, d in obj.items()}
    body["__qnlep_cache__"] = {"fingerprint": fingerprint}
    return body


def _split_partial_disk(raw):
    """Return (results_by_method, fingerprint_or_None). Legacy files have fp None."""
    if not raw:
        return {}, None
    data = dict(raw)
    meta = data.pop("__qnlep_cache__", None)
    fp = meta.get("fingerprint") if isinstance(meta, dict) else None
    out = {}
    for m, dk in data.items():
        out[m] = {int(k): v for k, v in dk.items()}
    return out, fp


def _current_nlc_fingerprint():
    """If any of these change, disk cache is ignored (no manual delete)."""
    nsbm = {k: int(N_STEPS_BY_METHOD[k]) for k in sorted(N_STEPS_BY_METHOD)}
    return {
        "run_seed": int(RUN_SEED),
        "n_steps": int(N_STEPS),
        "n_steps_by_method": nsbm,
        "n_qubits": int(N_QUBITS),
        "n_layers": int(N_LAYERS),
        "k_values": [int(k) for k in K_VALUES],
        "methods": list(QNLEP_METHOD_ORDER),
        "lr": float(LR),
        "r2_thresh": float(R2_THRESH),
        "n_train": int(N_TRAIN),
        "seed_strategy": "task_local_seed_v1",
    }


def load_partial_results():
    if not NLC_PARTIAL_JSON.exists():
        return {}
    with open(NLC_PARTIAL_JSON, encoding="utf-8") as f:
        raw = json.load(f)
    out, fp_disk = _split_partial_disk(raw)
    want = _current_nlc_fingerprint()
    if fp_disk != want:
        print(
            "[NLC partial] ignoring disk (fingerprint mismatch or legacy file).",
            f"Path={NLC_PARTIAL_JSON.resolve()}",
        )
        return {}
    return out


def save_partial_results(obj):
    fp = _current_nlc_fingerprint()
    payload = _partial_to_disk(obj, fp)
    with open(NLC_PARTIAL_JSON, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def _nlc_retrain_methods_as_tuple(x):
    """Accept (), list, tuple, or comma-separated str; never iterate a method name by char."""
    if x is None:
        return ()
    if isinstance(x, str):
        return tuple(m.strip() for m in x.split(",") if m.strip())
    return tuple(x)


def _nlc_task_seed(method_name, k):
    method_idx = QNLEP_METHOD_ORDER.index(method_name)
    return int(RUN_SEED) * 10_000 + method_idx * 100 + int(k)


_env_full = os.environ.get("QNLEP_NLC_IGNORE_PARTIAL", "").strip().lower() in ("1", "true", "yes")
_env_meth = _nlc_retrain_methods_as_tuple(os.environ.get("QNLEP_NLC_RETRAIN_METHODS", ""))
_cfg_full = bool(_nlc_config_get("NLC_FORCE_FULL_RETRAIN", False))
_cfg_meth = _nlc_retrain_methods_as_tuple(_nlc_config_get("NLC_FORCE_RETRAIN_METHODS", ()))
_load_disk = bool(_nlc_config_get("NLC_LOAD_PARTIAL_FROM_DISK", True))
_force_full = _cfg_full or _env_full
_drop = set(_cfg_meth) | set(_env_meth)

if _force_full:
    results = {}
elif not _load_disk:
    results = {}
    print("[NLC partial] NLC_LOAD_PARTIAL_FROM_DISK=False — starting fresh (still saving after each k).")
else:
    results = load_partial_results()
for _m in _drop:
    results.pop(_m, None)
print(
    f"[NLC partial] path={NLC_PARTIAL_JSON.resolve()}  "
    f"force_full={_force_full}  load_disk={_load_disk}  drop={sorted(_drop) if _drop else '—'}"
)

for method_name, circuit in METHODS.items():
    if method_name not in results:
        results[method_name] = {}
    for k in K_VALUES:
        if k in results[method_name] and 'r2' in results[method_name][k]:
            print(f"{method_name:20s}  k={k:2d}  R²={results[method_name][k]['r2']:.3f}  (cached)")
            continue

        y_tr = np.sin(k * x_train)
        y_ev = np.sin(k * x_eval)

        n_steps_m = N_STEPS_BY_METHOD.get(method_name, N_STEPS)
        weights, history = train_on_task(
            method_name,
            circuit,
            x_train,
            y_tr,
            n_steps=n_steps_m,
            seed=_nlc_task_seed(method_name, k),
        )
        preds = evaluate(circuit, weights, x_eval)
        r2 = r2_score(y_ev, preds)

        results[method_name][k] = {'r2': r2, 'preds': preds.tolist(), 'history': history}
        save_partial_results(results)
        print(f"{method_name:20s}  k={k:2d}  R²={r2:.3f}")


## 4. Compute NLC scores

In [ ]:
def k_star_at_threshold(results_for_method, k_values=K_VALUES, thresh=R2_THRESH):
    """Largest k in k_values with R² >= thresh; None if none pass."""
    k_star = None
    for k in k_values:
        if results_for_method[k]['r2'] >= thresh:
            k_star = k
    return k_star


def compute_nlc(results_for_method, k_values=K_VALUES, thresh=R2_THRESH):
    r2_values = [float(results_for_method[k]['r2']) for k in k_values]
    return continuous_frequency_score(r2_values, k_values)


nlc_scores = {m: compute_nlc(results[m]) for m in METHODS}
nlc_detail = {}
for m in METHODS:
    r2_values = [float(results[m][k]['r2']) for k in K_VALUES]
    legacy_k_star = legacy_k_star_from_curve(r2_values, K_VALUES, R2_THRESH)
    legacy_score = None if legacy_k_star is None else float(np.log2(legacy_k_star) / np.log2(K_MAX))
    nlc_detail[m] = {
        'nlc': nlc_scores[m],
        'score_definition': 'continuous_area_under_positive_r2_vs_log2_frequency',
        'legacy_k_star': legacy_k_star,
        'legacy_threshold_score': legacy_score,
        'r2_by_k': {str(k): float(results[m][k]['r2']) for k in K_VALUES},
        'positive_r2_by_k': {str(k): positive_r2_score(results[m][k]['r2']) for k in K_VALUES},
        'r2_threshold': R2_THRESH,
        'k_max': K_MAX,
    }

print("\nContinuous NLC scores (area under positive-R² vs log-frequency curve):")
for m, s in nlc_scores.items():
    ks = nlc_detail[m]['legacy_k_star']
    print(f"  {m:20s}  NLC = {s:.3f}   legacy_k* = {ks}")


## 5. Visualizations

### 5a. Fit curves — method × frequency grid

With many methods this figure is tall; scroll the notebook or open the saved PDF. All rows use the same `METHODS` dict as the sweep.

In [ ]:
n_methods = len(METHODS)
row_h = min(3.0, 18 / max(n_methods, 1))
fig, axes = plt.subplots(n_methods, len(K_VALUES),
                         figsize=(3.5 * len(K_VALUES), row_h * n_methods),
                         sharex=True, sharey=True)

for row, method_name in enumerate(METHODS):
    for col, k in enumerate(K_VALUES):
        ax = axes[row, col] if n_methods > 1 else axes[col]
        r = results[method_name][k]
        y_t = np.sin(k * x_eval)

        ax.plot(x_eval, y_t, 'k--', lw=1.5, label='target')
        ax.plot(x_eval, r['preds'], 'C0', lw=1.5, label='predicted')
        ax.set_title(f"k={k}  R²={r['r2']:.2f}", fontsize=9)
        ax.axhline(0, color='grey', lw=0.5, ls=':')

        if col == 0:
            ax.set_ylabel(method_name.replace('_', '\n'), fontsize=8)

leg_ax = axes[0, 0] if n_methods > 1 else axes[0]
leg_ax.legend(fontsize=7)
fig.suptitle('NLC — Fit quality per method and frequency', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nlc_fit_grid.pdf', bbox_inches='tight')
plt.show()


### 5b. R² heatmap (method × k)

In [ ]:
method_names = list(METHODS.keys())
n_methods = len(method_names)
r2_matrix = np.array([[results[m][k]['r2'] for k in K_VALUES] for m in method_names])

# R² can be negative (worse than predicting the mean); vmin=0 would clip that in colour.
r2_lo = float(np.min(r2_matrix))
v_min = min(0.0, r2_lo - 0.05)

fig, ax = plt.subplots(figsize=(8, max(3, 2.0 + 0.55 * n_methods)))
im = ax.imshow(r2_matrix, vmin=v_min, vmax=1.0, cmap='RdYlGn', aspect='auto')

ax.set_xticks(range(len(K_VALUES)))
ax.set_xticklabels([f'k={k}' for k in K_VALUES])
ax.set_yticks(range(n_methods))
ax.set_yticklabels(method_names)

for i in range(n_methods):
    for j in range(len(K_VALUES)):
        ax.text(j, i, f"{r2_matrix[i, j]:.2f}", ha='center', va='center', fontsize=9)

plt.colorbar(im, ax=ax, label='R²  (negative = worse than mean predictor)')
ax.set_title(f'R² heatmap  (NLC threshold = {R2_THRESH})', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nlc_r2_heatmap.pdf', bbox_inches='tight')
plt.show()


### 5c. NLC bar chart

In [ ]:
method_names = list(METHODS.keys())
n_methods = len(method_names)
fig_w = max(7.0, 0.9 * n_methods + 3)
fig, ax = plt.subplots(figsize=(fig_w, 3.8))
heights = [0.0 if nlc_scores[m] is None else nlc_scores[m] for m in method_names]
bars = ax.bar(method_names, heights, color='steelblue', edgecolor='k')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Continuous NLC score')
ax.set_title('Continuous Fourier-capacity score per method')
ax.tick_params(axis='x', rotation=22)
for lab in ax.get_xticklabels():
    lab.set_ha('right')
ax.axhline(1.0, color='green', ls='--', lw=1, label='perfect')
ax.legend(fontsize=8)
for bar, m in zip(bars, method_names):
    label = '—' if nlc_scores[m] is None else f"{nlc_scores[m]:.2f}"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            label, ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nlc_bar.pdf', bbox_inches='tight')
plt.show()


## 6. Export NLC scores
Save `nlc_scores.json` for notebook 09 (flat floats / null). Save `nlc_detail.json`
with `k_star`, per-`k` R², and thresholds so NLC = 0 is interpretable.

In [ ]:
with open(RESULTS_DIR / 'nlc_scores.json', 'w', encoding='utf-8') as f:
    json.dump(nlc_scores, f, indent=2)
with open(RESULTS_DIR / 'nlc_detail.json', 'w', encoding='utf-8') as f:
    json.dump(nlc_detail, f, indent=2)
with open(RESULTS_DIR / 'nlc_metric_meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'score_definition': 'continuous_area_under_positive_r2_vs_log2_frequency',
        'k_values': K_VALUES,
        'r2_threshold_legacy': R2_THRESH,
        'n_train': N_TRAIN,
        'n_eval': N_EVAL,
        'n_steps': N_STEPS,
        'n_steps_by_method': N_STEPS_BY_METHOD,
        'run_seed': RUN_SEED,
    }, f, indent=2)
print('Saved to', RESULTS_DIR / 'nlc_scores.json', ',', RESULTS_DIR / 'nlc_detail.json', 'and', RESULTS_DIR / 'nlc_metric_meta.json')
